# TextCNN Baseline

Run this notebook from the project root or from the `baselines/` folder. It uses the shared benchmark code, writes artifacts under `artefacts/`, and evaluates `textcnn` on the fixed split.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "baselines").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/home/ilya/ML/NLP/project')

In [2]:
MODEL_NAME = "textcnn"
MAX_SAMPLES = None  # set to a small integer, e.g. 3000, for debugging
TOP_GENRES = 15
EPOCHS = 3
BATCH_SIZE = 16
TFIDF_MAX_FEATURES = 100_000

In [3]:
import pandas as pd

from baselines.config import BaselineConfig
from baselines.data_utils import prepare_data
from baselines.run_all import finalize_model_result
from baselines.neural_baselines import train_neural_baseline

config = BaselineConfig(
    max_samples=MAX_SAMPLES,
    top_genres=TOP_GENRES,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    tfidf_max_features=TFIDF_MAX_FEATURES,
)

/home/ilya/ML/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
bundle = prepare_data(config)
bundle.stats

Found candidate tabular files:
  /home/ilya/ML/NLP/project/data/tmdb_movies_2021_2025.csv (80.5 MB)
  /home/ilya/ML/NLP/project/data/tmdb_movies_2021_2025.parquet (50.9 MB)
Choosing largest file by default: /home/ilya/ML/NLP/project/data/tmdb_movies_2021_2025.csv
Prepared data: train=87575, val=34470, test=32986, labels=15, split=temporal_train_le_2023_val_2024_test_2025


{'source_path': '/home/ilya/ML/NLP/project/data/tmdb_movies_2021_2025.csv',
 'detected_columns': {'title': 'title',
  'overview': 'overview',
  'genres': 'genres',
  'release_date': 'release_date',
  'id': 'tmdb_id'},
 'rows_before_filtering': 232586,
 'rows_after_overview_filter': 193927,
 'rows_after_genre_parse': 155678,
 'rows_after_top_genre_filter': 155031,
 'selected_genre_labels': ['Drama',
  'Documentary',
  'Comedy',
  'Horror',
  'Thriller',
  'Animation',
  'Romance',
  'Music',
  'Action',
  'Crime',
  'Fantasy',
  'Science Fiction',
  'Mystery',
  'Family',
  'TV Movie'],
 'per_label_frequency': {'Drama': 54784,
  'Documentary': 43107,
  'Comedy': 29457,
  'Horror': 18098,
  'Thriller': 15386,
  'Animation': 12723,
  'Romance': 10576,
  'Music': 8382,
  'Action': 7238,
  'Crime': 6677,
  'Fantasy': 6339,
  'Science Fiction': 6141,
  'Mystery': 6245,
  'Family': 5088,
  'TV Movie': 3842},
 'train_per_label_frequency': {'Drama': 30116,
  'Documentary': 25421,
  'Comedy': 15

In [5]:
assert bundle.y_train.shape[1] == len(bundle.label_names)
assert bundle.y_val.shape[1] == len(bundle.label_names)
assert bundle.y_test.shape[1] == len(bundle.label_names)
assert {"sample_id", "text", "labels_list"}.issubset(bundle.train_df.columns)
assert bundle.train_df["text"].str.len().min() >= config.min_overview_chars

print("labels:", bundle.label_names)
print("train/val/test:", bundle.y_train.shape, bundle.y_val.shape, bundle.y_test.shape)

labels: ['Drama', 'Documentary', 'Comedy', 'Horror', 'Thriller', 'Animation', 'Romance', 'Music', 'Action', 'Crime', 'Fantasy', 'Science Fiction', 'Mystery', 'Family', 'TV Movie']
train/val/test: (87575, 15) (34470, 15) (32986, 15)


In [6]:
result = train_neural_baseline("textcnn", bundle, config)
metrics = finalize_model_result(result, bundle, config)
metrics

textcnn epoch 1: loss=0.2490, val_macro_f1=0.3714


textcnn epoch 2: loss=0.2195, val_macro_f1=0.3992


textcnn epoch 3: loss=0.2026, val_macro_f1=0.4066


{'model': 'textcnn',
 'micro_f1': 0.5252354176546304,
 'macro_f1': 0.40247707095944835,
 'weighted_f1': 0.5246183035524128,
 'samples_f1': 0.5485052301929159,
 'precision_micro': 0.46532425051766046,
 'recall_micro': 0.6028536993118464,
 'hamming_loss': 0.11330867640817316,
 'precision_at_1': 0.6270842175468381,
 'precision_at_3': 0.35480708987651327,
 'recall_at_3': 0.7483973118947047,
 'subset_accuracy': 0.24777178196810767,
 'average_precision_micro': 0.5549868523210066}

In [7]:
prediction_path = PROJECT_ROOT / "artefacts" / "predictions" / f"{MODEL_NAME}_test_predictions.csv"
threshold_path = PROJECT_ROOT / "artefacts" / "thresholds" / f"{MODEL_NAME}_thresholds.json"
print(prediction_path)
print(threshold_path)
pd.read_csv(prediction_path).head()

/home/ilya/ML/NLP/project/artefacts/predictions/textcnn_test_predictions.csv
/home/ilya/ML/NLP/project/artefacts/thresholds/textcnn_thresholds.json


,sample_id,text,true_labels,predicted_labels,score_drama,score_documentary,score_comedy,score_horror,score_thriller,score_animation,score_romance,score_music,score_action,score_crime,score_fantasy,score_science_fiction,score_mystery,score_family,score_tv_movie
0,1052558,iPossessed [SEP] A group of celebrating friend...,Horror|Thriller,Horror|Thriller,0.267388,0.020499,0.090816,0.919588,0.324374,0.097083,0.007685,0.009307,0.088126,0.022836,0.089137,0.055820,0.075573,0.009408,0.003458
1,980477,Ne Zha 2 [SEP] After a catastrophic event leav...,Animation|Action|Fantasy,Drama|Comedy|Action,0.625775,0.011148,0.270766,0.078229,0.095926,0.044760,0.066491,0.004976,0.151092,0.036623,0.063190,0.076455,0.028573,0.067728,0.014848
2,1205229,Night of the Zoopocalypse [SEP] A wolf and mou...,Comedy|Horror|Animation|Science Fiction,Horror|Thriller|Action|Science Fiction,0.092434,0.013575,0.062279,0.850579,0.172162,0.061586,0.000920,0.005920,0.104031,0.007712,0.059745,0.262945,0.034073,0.010699,0.000616
3,1084199,Companion [SEP] During a weekend getaway at a ...,Horror|Thriller|Science Fiction,Comedy|Horror|Thriller|Action|Crime|Mystery,0.351456,0.003946,0.110370,0.640157,0.813369,0.008443,0.004798,0.006970,0.140988,0.449696,0.038054,0.160554,0.249761,0.003486,0.014817
4,1009640,Valiant One [SEP] With tensions between North ...,Thriller|Action,Drama,0.414834,0.369323,0.078384,0.056407,0.030358,0.098126,0.008824,0.007943,0.017064,0.025318,0.026835,0.021600,0.022511,0.014829,0.008494
